# M1 Validation: ManiSkill + Octo (Linux/Colab)

End-to-end M1 acceptance on a Google Colab Linux+CUDA VM (run from Cursor IDE, no `google.colab` imports).
Covers environment setup, Octo VLA server startup, the ManiSkill PickCube example, and the project test suite.

## 1 — Secrets & clone

In [ ]:
import getpass, os, subprocess, sys

os.environ['GITHUB_TOKEN']   = getpass.getpass('GitHub PAT (repo scope): ')
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

subprocess.run(
    ['git', 'clone', f"https://{os.environ['GITHUB_TOKEN']}@github.com/pmish11/Longor-Robotics.git", '/content/Longor'],
    check=True
)
os.chdir('/content/Longor')
sys.path.insert(0, '/content/Longor/src')

## 2 — Headless rendering env vars

In [ ]:
import os

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'
# Conservative default for Colab; remove or set to gpu on hosts with confirmed Vulkan support.
os.environ['MANISKILL_SIM_BACKEND'] = 'cpu'

## 3 — Install app deps

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', *args], check=True)

# On Linux+CUDA you would normally install CUDA PyTorch first; requirements-maniskill.txt
# pins CPU torch as the smoke-test baseline, which is fine for M1 wiring validation.
pip('install', '-U', 'pip')
pip('install', '-r', 'requirements.txt')
pip('install', '-r', 'requirements-maniskill.txt')

## 4 — Install VLA env (Octo)

In [ ]:
import subprocess, sys

VLA_PYTHON = '/content/Longor/.vla-venv/bin/python'

def vla_pip(*args):
    result = subprocess.run(
        [VLA_PYTHON, '-m', 'pip', *args],
        capture_output=True, text=True
    )
    if result.stdout:
        print(result.stdout[-3000:])
    if result.stderr:
        print(result.stderr[-3000:])
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, result.args)

# Colab (Debian/Ubuntu) doesn't ship python3-venv; virtualenv works without OS packages.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'virtualenv'], check=True)
if not os.path.exists('/content/Longor/.vla-venv'):
    subprocess.run([sys.executable, '-m', 'virtualenv', '/content/Longor/.vla-venv'], check=True)
else:
    print('.vla-venv already exists, skipping creation.')
vla_pip('install', '-U', 'pip')
vla_pip('install', 'setuptools', 'wheel')
# Use latest JAX — compatible with latest flax/optax/numpy. We patch Octo's KeyArray reference below.
vla_pip('install', '-U', 'jax[cuda12]')
import os
if not os.path.exists('/root/octo'):
    subprocess.run(['git', 'clone', 'https://github.com/octo-models/octo.git', '/root/octo'], check=True)
else:
    print('/root/octo already exists, skipping clone.')
vla_pip('install', '-e', '/root/octo')
# octo/requirements.txt pins versions that have no Python 3.12 wheels and fail to build from source.
# Skip it entirely; install packages not covered by Octo's setup.py with Py3.12-compatible versions.
vla_pip('install', 'numpy>=1.26', 'ml_dtypes>=0.4', 'Pillow', 'flax', 'tensorflow')
# dlimp pins tensorflow==2.15.0 (no Py3.12 wheel); install with --no-deps since TF is already installed
vla_pip('install', '--no-deps', 'git+https://github.com/kvablack/dlimp.git')
# Other Octo runtime deps likely not in setup.py
vla_pip('install', 'optax', 'distrax', 'wandb', 'tensorflow_datasets', 'tf_keras',
         'einops', 'orbax-checkpoint', 'huggingface_hub')
vla_pip('install', '-r', '/content/Longor/services/requirements-vla-octo-small-1.5-linux.txt')

## 5 — Patch Octo for JAX ≥ 0.4.24

In [ ]:
# jax.random.KeyArray was removed in JAX 0.4.24. Patch the one Octo file that references it.
_typing_path = '/root/octo/octo/utils/typing.py'
with open(_typing_path) as _f:
    _src = _f.read()
_patched = _src.replace('jax.random.KeyArray', 'jax.Array')
if _patched == _src:
    print('No KeyArray reference found — already patched or not present.')
else:
    with open(_typing_path, 'w') as _f:
        _f.write(_patched)
    print('Patched octo/utils/typing.py: jax.random.KeyArray → jax.Array')

## 6 — Write `.env`

In [ ]:
import os

overrides = {
    'OPENAI_API_KEY': os.environ['OPENAI_API_KEY'],
    'VLA_POLICY': 'octo-small-1.5',
    'VLA_PORT': '8765',
    'MANISKILL_ENV': 'PickCube-v1',
    'BACKEND': 'maniskill',
    'MANISKILL_SIM_BACKEND': 'cpu',
}

lines_out = []
seen_keys = set()
with open('/content/Longor/.env.example') as f:
    for line in f:
        stripped = line.rstrip('\n\r')
        if stripped and not stripped.startswith('#') and '=' in stripped:
            key, _, _ = stripped.partition('=')
            key = key.strip()
            if key in overrides:
                lines_out.append(f'{key}={overrides[key]}')
                seen_keys.add(key)
                continue
        lines_out.append(stripped)

for key, val in overrides.items():
    if key not in seen_keys:
        lines_out.append(f'{key}={val}')

with open('/content/Longor/.env', 'w') as f:
    f.write('\n'.join(lines_out) + '\n')
print('Wrote /content/Longor/.env')

## 6 — Start VLA server

In [ ]:
import subprocess, time, urllib.request, json

VLA_LOG = '/tmp/vla_server.log'
vla_python = '/content/Longor/.vla-venv/bin/python'

def _tail_log(n=40):
    try:
        with open(VLA_LOG) as f:
            lines = f.readlines()
        return ''.join(lines[-n:])
    except FileNotFoundError:
        return '(log file not created yet)'

# Octo blocks /health until load_pretrained finishes (HF download on first run can take 10+ min).
log = open(VLA_LOG, 'w', buffering=1)
vla_proc = subprocess.Popen(
    [vla_python, 'services/run_vla_server.py'],
    stdout=log, stderr=subprocess.STDOUT, cwd='/content/Longor'
)

POLL_SEC = 15
MAX_WAIT_SEC = 20 * 60
for i in range(MAX_WAIT_SEC // POLL_SEC):
    time.sleep(POLL_SEC)
    elapsed = (i + 1) * POLL_SEC

    if vla_proc.poll() is not None:
        print(_tail_log(60))
        raise RuntimeError(f'VLA server exited early (code {vla_proc.returncode}). Log above.')

    try:
        resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
        print('VLA server ready:', json.loads(resp.read()))
        break
    except Exception:
        print(f'Waiting... ({elapsed}s)')
        if elapsed % 60 == 0:
            print('--- recent log ---')
            print(_tail_log(15))
else:
    print(_tail_log(60))
    raise RuntimeError(f'VLA server did not come up in {MAX_WAIT_SEC // 60} minutes. Log above.')

## 7 — VLA server log (diagnostic)

In [ ]:
with open('/tmp/vla_server.log') as f:
    lines = f.readlines()
print(''.join(lines[-30:]))

## 8 — Run ManiSkill PickCube example

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'examples/maniskill_pickcube.py'],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=300
)
print(result.stdout[-3000:])
print(result.stderr[-1000:])
print('Exit code:', result.returncode)

## 9 — Run pytest

In [ ]:
import os, subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-m', 'not integration', '--tb=short', '-q'],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=300,
    env={**os.environ, 'PYTHONPATH': '/content/Longor/src'}
)
print(result.stdout[-4000:])
print(result.stderr[-1000:])
assert result.returncode == 0, f'pytest failed (exit {result.returncode})'

## 10 — Teardown

In [ ]:
vla_proc.terminate()
print('VLA server stopped.')